# Data exploration — offline engine sanity checks

Loads the dataset from `python -m data_pipeline.run_offline` (`outputs/task04_dataset.npz`)
and plots sanity checks (saved to `report/figures/`). The dataset includes the raw
MediaPipe landmark layer, the resampled IMU + recovered head orientation, the approximate
3D wrist targets, and the full 53-joint G1+Inspire trajectory (see `*.schema.json`).

1. approximate 3D wrist trajectory, 2. hand-present timeline, 3. waist-lean target,
4. IMU visualization (accel/gyro/head-orientation/waist-yaw).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
d = np.load('../outputs/task04_dataset.npz', allow_pickle=True)
t = d['timestamps']
print('frames', len(t), 'fps', float(d['fps']))
print('arrays:', len(d.files), '| actuated joints:', len(d['actuated_joint_names']))
print('present: left %.1f%%  right %.1f%%' % (d['present_left'].mean()*100, d['present_right'].mean()*100))

## 1. Approximate 3D wrist trajectory (depth is approximated from RGB)

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
for s, c in (('right', 'C0'), ('left', 'C1')):
    wt = d[f'wrist_target_{s}']
    ax[0].plot(t, wt[:, 0], c, label=f'{s} x (fwd)', alpha=.8)
    ax[0].plot(t, wt[:, 2], c, ls='--', label=f'{s} z (height)', alpha=.6)
    ax[1].plot(t, wt[:, 1], c, label=f'{s} y (lateral)', alpha=.8)
ax[0].set_ylabel('m'); ax[0].legend(fontsize=8, ncol=2)
ax[0].set_title('Approx 3D wrist target (RGB-only — depth approximated)')
ax[1].set_ylabel('m'); ax[1].set_xlabel('time (s)'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig('../report/figures/plot_wrist_trajectory.png', dpi=110); plt.show()

## 2. Hand-present timeline (gaps → per-arm stow)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 2.4))
ax.fill_between(t, 0, d['present_right'].astype(int), step='mid', alpha=.6, label=f"right ({d['present_right'].mean()*100:.0f}%)")
ax.fill_between(t, 1.2, 1.2+d['present_left'].astype(int), step='mid', alpha=.6, color='C1', label=f"left ({d['present_left'].mean()*100:.0f}%)")
ax.set_yticks([]); ax.set_xlabel('time (s)'); ax.set_title('Hand-present timeline (gaps → stow)')
ax.legend(fontsize=8, loc='upper right'); plt.tight_layout()
plt.savefig('../report/figures/plot_hand_presence.png', dpi=110); plt.show()

## 3. Waist-lean target from head IMU

In [ ]:
wl = np.rad2deg(d['waist_lean'])
fig, ax = plt.subplots(figsize=(9, 3))
for k, lbl in enumerate(['roll', 'pitch', 'yaw']):
    ax.plot(t, wl[:, k], label=lbl)
ax.set_ylabel('deg'); ax.set_xlabel('time (s)'); ax.legend(fontsize=8)
ax.set_title('Waist-lean target (tuned: roll/pitch locked 0, yaw-only)')
plt.tight_layout(); plt.savefig('../report/figures/plot_waist_lean.png', dpi=110); plt.show()

## 4. IMU visualization (accel, gyro, recovered head orientation, applied waist-yaw)

In [ ]:
fig, ax = plt.subplots(4, 1, figsize=(10, 9), sharex=True)
for k, l in enumerate('xyz'): ax[0].plot(t, d['imu_accel'][:, k], label=f'a{l}', lw=.8)
ax[0].set_ylabel('accel\n(m/s²)'); ax[0].legend(ncol=3, fontsize=8)
ax[0].set_title('Head IMU resampled to video frames + recovered orientation')
for k, l in enumerate('xyz'): ax[1].plot(t, d['imu_gyro'][:, k], label=f'g{l}', lw=.8)
ax[1].set_ylabel('gyro\n(rad/s)'); ax[1].legend(ncol=3, fontsize=8)
for arr, lab in (('head_roll', 'roll'), ('head_pitch', 'pitch'), ('head_yaw', 'yaw')):
    ax[2].plot(t, np.rad2deg(d[arr]), label=lab)
ax[2].set_ylabel('head orient\n(deg)'); ax[2].legend(ncol=3, fontsize=8)
ax[3].plot(t, np.rad2deg(d['waist_yaw_applied']), color='C3', label='waist_yaw applied')
ax[3].axhline(0, color='k', lw=.5, ls=':')
ax[3].set_ylabel('waist yaw\n(deg)'); ax[3].set_xlabel('time (s)'); ax[3].legend(fontsize=8)
plt.tight_layout(); plt.savefig('../report/figures/imu_visualization.png', dpi=110); plt.show()